In [1]:
from typing import Any, Final

import numpy as np
import pandas as pd

In [2]:
survey = pd.read_csv("res/survey/wide.csv")
survey

,pid,gender,age,education,employment,income,car_own,home,dest1,purp1,...,mode3,time3,dest4,purp4,mode4,time4,dest5,purp5,mode5,time5
0,101,1: male,28.0,4: master or phd,4: active,2: 750-1500,0: no,2.0,15,1: work,...,8: escooter,22.0,2.0,2: return home,8: escooter,2.0,NaN,NaN,NaN,NaN
1,102,0: feamale,28.0,2: high school,3: student,0: no income,1: yes,12.0,12,3: education,...,1: car,21.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,103,0: feamale,27.0,3: bachelor,4: active,2: 750-1500,1: yes,15.0,15,1: work,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,104,0: feamale,18.0,4: master or phd,4: active,2: 750-1500,0: no,7.0,30,1: work,...,4: train,21.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,105,1: male,18.0,4: master or phd,4: active,3: 1500-2500,1: yes,22.0,35,5: recreation,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
508,609,1: male,37.0,3: bachelor,4: active,NaN,1: yes,16.0,1,1: work,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
509,610,0: feamale,36.0,3: bachelor,4: active,NaN,1: yes,11.0,32,1: work,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
510,611,0: feamale,53.0,3: bachelor,4: active,NaN,1: yes,28.0,28,1: work,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
511,612,1: male,21.0,4: master or phd,4: active,NaN,1: yes,12.0,22,4: market,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
MIN_TRIP_INDEX: Final[int] = 1
MAX_TRIP_INDEX: Final[int] = 5

# Require a valid home zone for trip chain construction.
survey = survey.dropna(subset="home")

trip_data = []
for record in survey.itertuples(index=False):
    pid = record.pid

    # Construct the trip chain.
    prev_dzone = record.home
    for i in range(MIN_TRIP_INDEX, MAX_TRIP_INDEX + 1):
        curr_dzone = getattr(record, f"dest{i}")
        if pd.isna(curr_dzone):
            # There are no more reported trips.
            break
        trip_data.append(
            {
                "pid": pid,
                "ozone": prev_dzone,
                "dzone": curr_dzone,
                "purp": getattr(record, f"purp{i}"),
                "mode": getattr(record, f"mode{i}"),
                "time": getattr(record, f"time{i}"),
            }
        )
        prev_dzone = curr_dzone

trips = pd.DataFrame(trip_data)
trips

,pid,ozone,dzone,purp,mode,time
0,101,2.0,15.0,1: work,4: train,9.0
1,101,15.0,2.0,1: work,7: walk,18.0
2,101,2.0,1.0,5: recreation,8: escooter,22.0
3,101,1.0,2.0,2: return home,8: escooter,2.0
4,102,12.0,12.0,3: education,1: car,11.0
...,...,...,...,...,...,...
1342,611,28.0,28.0,2: return home,1: car,19.0
1343,612,12.0,22.0,4: market,1: car,9.0
1344,612,22.0,1.0,5: recreation,1: car,18.0
1345,613,26.0,26.0,5: recreation,7: walk,21.0


In [4]:
trips["purp"] = (
    trips["purp"]
    .str.replace(r"^\d+:\s*", "", regex=True)
    .replace("return home", "home")
)
trips["mode"] = trips["mode"].str.replace(r"^\d+:\s*", "", regex=True)
trips

,pid,ozone,dzone,purp,mode,time
0,101,2.0,15.0,work,train,9.0
1,101,15.0,2.0,work,walk,18.0
2,101,2.0,1.0,recreation,escooter,22.0
3,101,1.0,2.0,home,escooter,2.0
4,102,12.0,12.0,education,car,11.0
...,...,...,...,...,...,...
1342,611,28.0,28.0,home,car,19.0
1343,612,12.0,22.0,market,car,9.0
1344,612,22.0,1.0,recreation,car,18.0
1345,613,26.0,26.0,recreation,walk,21.0


In [5]:
## Temporal Domain Standardization

# Trips that depart "earlier" than their predecessors occur on the following day.
occurs_on_next_day = trips.groupby("pid")["time"].diff() < 0
trips["time"] += (
    # Current Day (i.e., 0, 1,...)
    occurs_on_next_day.groupby(trips["pid"]).cumsum() * 24
)
trips

,pid,ozone,dzone,purp,mode,time
0,101,2.0,15.0,work,train,9.0
1,101,15.0,2.0,work,walk,18.0
2,101,2.0,1.0,recreation,escooter,22.0
3,101,1.0,2.0,home,escooter,26.0
4,102,12.0,12.0,education,car,11.0
...,...,...,...,...,...,...
1342,611,28.0,28.0,home,car,19.0
1343,612,12.0,22.0,market,car,9.0
1344,612,22.0,1.0,recreation,car,18.0
1345,613,26.0,26.0,recreation,walk,21.0


In [6]:
## Departure Time Window Reconstruction

TIME_WINDOWS: Final[tuple[tuple[int, int], ...]] = (
    (5, 8),
    (8, 11),
    (11, 14),
    (14, 17),
    (17, 20),
    (20, 23),
)

MIN_TIMES: Final[np.ndarray[tuple[Any,], np.dtype[np.int64]]] = np.array(
    [time[0] for time in TIME_WINDOWS]
)
MAX_TIMES: Final[np.ndarray[tuple[Any,], np.dtype[np.int64]]] = np.array(
    [time[1] for time in TIME_WINDOWS]
)

# --------------------------------------------------

times = trips["time"]

# Infer the current day and hour.
days = np.floor(times / 24)
hours = np.mod(times, 24)

# Find the start time of the corresponding window.
min_time_index = np.searchsorted(MIN_TIMES, hours, side="right") - 1
# Clamp times between 23:00 and 05:00 to 05:00.
# These times belong to a special window that begins at 23:00 on the current day and ends at 05:00 on the following day.
# This window is handled separately below.
min_time_index = np.clip(min_time_index, 0, len(MIN_TIMES) - 1)

# Construct the window on the current day.
hour_offsets = days * 24
min_time = MIN_TIMES[min_time_index] + hour_offsets
max_time = MAX_TIMES[min_time_index] + hour_offsets

# --------------------------------------------------

# Overnight Window - Current Day (23:00--23:59)
is_curr_day = hours >= 23
min_time = np.where(is_curr_day, 23 + hour_offsets, min_time)
max_time = np.where(is_curr_day, 29 + hour_offsets, max_time)

# Overnight Window - Following Day (00:00--04:59)
is_next_day = hours < 5
# 23:00 on the CURRENT day = 23 + (DAYS - 1) * 24 = -1 + OFFSETS
min_time = np.where(is_next_day, -1 + hour_offsets, min_time)
# 05:00 on the FOLLOWING day = 5 + DAYS * 24 = 5 + OFFSETS
max_time = np.where(is_next_day, 5 + hour_offsets, max_time)

trips["min_time"] = np.maximum(min_time, 0)
trips["max_time"] = max_time

trips = trips.drop(columns=["time"])
trips

,pid,ozone,dzone,purp,mode,min_time,max_time
0,101,2.0,15.0,work,train,8.0,11.0
1,101,15.0,2.0,work,walk,17.0,20.0
2,101,2.0,1.0,recreation,escooter,20.0,23.0
3,101,1.0,2.0,home,escooter,23.0,29.0
4,102,12.0,12.0,education,car,11.0,14.0
...,...,...,...,...,...,...,...
1342,611,28.0,28.0,home,car,17.0,20.0
1343,612,12.0,22.0,market,car,8.0,11.0
1344,612,22.0,1.0,recreation,car,17.0,20.0
1345,613,26.0,26.0,recreation,walk,20.0,23.0


In [7]:
trips.to_csv("res/survey/long/trips.csv", index=False)